# 🧩 OOP in Python — Part 2: Practice & Deeper Concepts

This notebook builds two full mini-projects (a simple ATM, then a private/secure ATM) and then digs into the deeper mechanics of Python OOP: real privacy conventions, getters/setters, reference semantics, static vs instance variables, and object relationships (Aggregation, Inheritance).

## 📑 Table of Contents

- [🎯 Learning Goal](#learning-goal)
- [🤔 What's Different in Part 2?](#whats-different)
- [🧠 Key Idea](#key-idea)
- [📚 Important Terms](#important-terms)
- [1. 🏧 Building an ATM (Simple Version)](#atm-simple)
- [2. 🔢 A `Fraction` Class — Operator Overloading](#fraction-class)
- [3. 🔒 Encapsulation with Name Mangling — a Private ATM](#encapsulation)
  - [3.1 The "Single Underscore" Convention](#single-underscore)
  - [3.2 Name Mangling (`__variable`)](#name-mangling)
  - [3.3 Full Reflection and Introspection](#reflection)
- [4. 🔑 Getter and Setter in Python](#getter-setter)
- [5. 🔗 Reference Variables in Python](#reference-variables)
  - [5.1 Comparing to Lists and Tuples](#mutable-vs-immutable)
- [6. 📦 Collections of Objects](#collections-of-objects)
- [7. 🧮 Static (Class) Variables vs Instance Variables](#static-vs-instance)
  - [7.1 A Private Static Counter with a Static Method](#private-static-counter)
- [8. 🧩 Class Relationships — Aggregation ("Has-A")](#aggregation)
- [9. 🌳 Class Relationships — Inheritance ("Is-A")](#inheritance)
  - [9.1 Types of Inheritance (Single, Multilevel, Hierarchical, Multiple)](#inheritance-types)
  - [9.2 Method Resolution Order (MRO) and the Diamond Problem](#mro)
  - [9.3 `super()` and Method Overriding](#super-overriding)
  - [9.4 `isinstance()` and `issubclass()`](#isinstance-issubclass)
- [10. 🎭 Polymorphism ("Many Forms")](#polymorphism)
  - [10.1 Duck Typing — Polymorphism Without Inheritance](#polymorphism)
  - [10.2 Method Overloading — Why Python Doesn't Have It](#method-overloading)
  - [10.3 Method Overriding as Polymorphism (via Inheritance)](#overriding-as-polymorphism)
- [11. 🪄 Dunder Methods ("Magic Methods")](#dunder-methods)
  - [11.1 Common Dunder Methods in Action](#dunder-methods)
  - [11.2 The `__eq__` / `__hash__` Gotcha](#eq-hash-gotcha)
- [12. 🎭 Abstraction — Formalizing the Interface](#abstraction)
  - [12.1 The Template Method Pattern](#abstraction-template-method)
  - [12.2 Abstract Properties](#abstraction-properties)
- [⚠ Common Misconceptions](#misconceptions)
- [🔍 Interview Questions](#interview-questions)
- [🎯 Key Takeaways](#key-takeaways)

> 💡 Click any link above to jump straight to that section.

<a id="learning-goal"></a>
## 🎯 Learning Goal

By the end of this notebook you should understand:
- How to build a stateful, menu-driven class (the ATM) using instance attributes and a loop
- Why Python has **no true private variables** — only naming *conventions* (`_x`) and **name mangling** (`__x`)
- The difference between a traditional getter/setter and the more Pythonic `@property` approach
- Why passing an object into a function passes a **reference**, and how that affects mutation
- The difference between a **class variable** (shared) and an **instance variable** (per-object)
- What **Aggregation** ("Has-A") and **Inheritance** ("Is-A") relationships look like in code

<a id="whats-different"></a>
## 🤔 What's Different in Part 2? (Real-life analogy)

Part 1 taught you what a class *is*. Part 2 is about what happens when objects **talk to each other** and **live inside a running program**.

Think of a bank vault 🏦: the outer door (public methods) is what customers use. The inner mechanism (private attributes like `__pin`, `__balance`) is deliberately hidden — Python doesn't *physically* lock it (unlike Java's `private`), it just renames the lock (**name mangling**) and trusts you not to pick it. That's the "consenting adults" philosophy you'll see quoted below.

<a id="key-idea"></a>
### 🧠 Key Idea

- Python has **no enforced privacy**. `_x` is a *convention* ("please don't touch this from outside"); `__x` triggers **name mangling** (`_ClassName__x`), which mostly just makes accidental access harder, not impossible.
- `@property` lets you call a method like an attribute (`obj.balance` instead of `obj.get_balance()`) while still running validation code behind the scenes.
- Objects are passed to functions **by reference** — mutating the object *inside* the function affects the original object outside it too. Reassigning the parameter name does not.
- A **class variable** is shared by every instance (great for counters); an **instance variable** belongs to one object only.
- **Aggregation** ("Has-A"): one class holds another class as an attribute (e.g. `Customer` has an `Address`). **Inheritance** ("Is-A"): one class extends another and gets all its methods for free (e.g. `Student` is a `User`).

<a id="important-terms"></a>
### 📚 Important Terms

| Term | Simple Meaning | Example |
|---|---|---|
| `_variable` | Convention meaning "internal use only" (not enforced) | `self._balance` |
| `__variable` | Triggers name mangling to `_ClassName__variable` | `self.__pin` |
| Name Mangling | Python renaming `__x` internally to make outside access awkward | `self.__vault_code` → `_BankAccount__vault_code` |
| `@property` | Decorator that turns a method into a read-like attribute | `@property def balance(self):` |
| `@x.setter` | Decorator that lets `obj.x = value` run custom validation | `@balance.setter` |
| Reference | A variable that points to an object in memory, not a copy of it | `cust = BankAccount(...)` |
| `id()` | Returns an object's memory address (identity) | `id(cust)` |
| Class Variable | Shared across all instances of a class | `Customer.counter` |
| Instance Variable | Unique to one object | `self.name` |
| Aggregation | "Has-A" relationship — one class contains another as data | `Customer.address` |
| Inheritance | "Is-A" relationship — a subclass reuses a parent class's methods | `class Student(User):` |

<a id="atm-simple"></a>
## 1. 🏧 Building an ATM (Simple Version)

A menu-driven ATM machine that runs in a loop, taking input until the user chooses to exit. This shows a class that manages **state** (`pin`, `balance`) across many method calls.

In [1]:
class AtmMachine:
    
    def __init__(self):
        self.pin = ""
        self.balance = 0
        self.menu()
    
    def menu(self):
        while True:
            choice = input(
                """
            Please choose any option
            1. Press 0 to add balance
            2. Press 1 to create pin
            3. Press 2 to change pin
            4. Press 3 to check balance
            5. Press 4 to withdraw
            6. Press any other key to exit
                """
                )
            
            if choice == "0":
                self.add_balance()
            elif choice == "1":
                self.create_pin()
            elif choice == "2":
                self.change_pin()
            elif choice == "3":
                self.check_balance()
            elif choice == "4":
                self.withdraw()
                pass
            else:
                print("Thank you for using the ATM. Goodbye!")
                break 
        
        
    def create_pin(self):
        self.user_pin = input("Enter your pin: ")
        self.pin = self.user_pin
        print("PIN created successfully!")
        
    def add_balance(self):
        amount = int(input("Enter the amount you want to add: "))
        self.balance += amount
        print(f"{amount} is added to your account")
        print("Current Balance: ", self.balance)
    
    def check_balance(self):
        print("Your Current balance is:", self.balance)
    
    def change_pin(self):
        old_pin = input("Enter your current pin: ")
        if old_pin == self.pin:
            self.pin = input("Enter your new pin: ")
            print("PIN changed successfully!")
        else:
            print("Incorrect PIN!")
            self.change_pin()
        
    def withdraw(self):
        amount = int(input("Enter amount to withdraw: "))
        if amount <= self.balance:
            self.balance -= amount
            print(f"{amount} withdrawn successfully.")
        else:
            print("Insufficient balance!")
        

**Note:** `self.menu()` is called at the very end of `__init__`, so simply creating the object (`obj = AtmMachine()`) immediately starts the interactive menu loop — there's no separate "start" method to call.

**Gotcha:** Nothing stops you from calling `withdraw()` or `check_balance()` before ever calling `create_pin()` — `self.pin` starts as `""` and `self.balance` starts as `0`, so those methods will just silently operate on the defaults rather than warning you no PIN has been set yet.

In [2]:
obj = AtmMachine()

Thank you for using the ATM. Goodbye!


**Note:** This cell is **interactive** — running it will prompt you for input in the notebook. The pattern below in Section 2 fixes the biggest weakness here: **anyone can call `add_balance()`, `withdraw()`, or `check_balance()` without ever entering a PIN**, because none of those methods check `self.pin`. Section 2 locks that down.

<a id="fraction-class"></a>
## 2. 🔢 A `Fraction` Class — Operator Overloading

Python lets you redefine what `+`, `-`, `*`, `/` mean for your own objects by implementing "dunder" (double-underscore) methods like `__add__`, `__sub__`, `__mul__`, `__truediv__`, and `__str__`.

In [3]:
class Fraction:
    
    def __init__(self, n, d):
        self.num = n
        self.den = d

        
    def __str__(self):
        # return "Hello"
        return "{}/{}".format(self.num, self.den)
    
    def __add__(self, other):
        temp_num = self.num * other.den + other.num * self.den
        temp_den = self.den * other.den
        
        return "{}/{}".format(temp_num, temp_den)

    def __sub__(self, other):
        temp_num = self.num * other.den - other.num * self.den
        temp_den = self.den * other.den
        
        return "{}/{}".format(temp_num, temp_den)
    
    def __mul__(self, other):
        temp_num = self.num * other.num
        temp_den = self.den * other.den
        
        return "{}/{}".format(temp_num, temp_den)
    
    def __truediv__(self, other):
        temp_num = self.num * other.den
        temp_den = self.den * other.num
        
        return "{}/{}".format(temp_num, temp_den)
        

x = Fraction(3, 4)
print(x)

y = Fraction(5, 6)
print(y)

print(x+y)
print(x-y)
print(x*y)
print(x/y)

3/4
5/6
38/24
-2/24
15/24
18/20


**Note:** `__str__` controls what `print(x)` displays — without it, `print(x)` would show something unhelpful like `<__main__.Fraction object at 0x...>`.

**Gotcha:** `__add__`, `__sub__`, `__mul__`, and `__truediv__` all `return` a **plain string** (via `"{}/{}".format(...)`), not a new `Fraction` object. That means `x + y` gives you back a string, and you can print it fine, but you **cannot chain operations** — `(x + y) + z` would fail, because a string doesn't have `__add__` defined the way `Fraction` does (string `+` would just concatenate text instead). Also, the results aren't simplified — `x*y` above gives `15/24` instead of the reduced `5/8`. A more complete version would `return Fraction(temp_num, temp_den)` and reduce using `math.gcd`. See the extra example below for a fixed version.

### Extra Example — Fixing `Fraction` to Support Chaining

Here's the same class, but `__add__` now returns an actual `Fraction` object (and reduces it using `math.gcd`), so operations can be chained: `(x + y) + z` works correctly.

In [4]:
import math

class FractionV2:
    
    def __init__(self, n, d):
        self.num = n
        self.den = d
    
    def _reduce(self, n, d):
        g = math.gcd(n, d)
        return FractionV2(n // g, d // g)
    
    def __str__(self):
        return "{}/{}".format(self.num, self.den)
    
    def __add__(self, other):
        temp_num = self.num * other.den + other.num * self.den
        temp_den = self.den * other.den
        return self._reduce(temp_num, temp_den)
    
    def __mul__(self, other):
        temp_num = self.num * other.num
        temp_den = self.den * other.den
        return self._reduce(temp_num, temp_den)

x = FractionV2(1, 4)
y = FractionV2(1, 4)
z = FractionV2(1, 2)

print(x + y)        # 1/2  (reduced from 2/4)
print((x + y) + z)  # 1/1  (chaining works because __add__ returns a FractionV2)
print(x * y)        # 1/16


1/2
1/1
1/16


**Note:** Compare this to the original `Fraction` above — because `__add__` now returns `FractionV2` (via `self._reduce(...)`), the result can be added *again* in `(x + y) + z`. This is the standard fix for the "operator overloading returns a plain value instead of a new object" gotcha.

<a id="encapsulation"></a>
# Encapsulation
### In Python, there is a famous saying among developers: "We are all consenting adults here." This philosophy is the core reason why Python does not enforce true privacy. Unlike languages like Java or C++, which have strict access modifiers (private, protected, public) enforced by the compiler, Python relies on convention, trust, and transparency.

## 3. 🔒 Encapsulation with Name Mangling — a Private ATM

Prefixing an attribute or method with `__` (double underscore) triggers **name mangling**: Python internally renames `self.__pin` to `self._ClassName__pin`. This doesn't make it truly private, but it does prevent *accidental* access and name clashes in subclasses.

In [5]:
## Encapsulation Use __ in front of data properties or method properties to hide and restrict the user to access.

class AtmMachine:
    
    def __init__(self):
        self.__pin = ""
        self.__balance = 0
        self.__menu()
    
    def __menu(self):
        while True:
            choice = input(
                """
            Please choose any option
            1. Press 1 to create pin
            2. Press 2 to add balance
            3. Press 3 to withdraw
            4. Press 4 to check balance
            5. Press 5 to change pin
            6. Press any other key to exit
                """
                )
          
            if choice == "1":  #  Press 1 to create pin
                self.create_pin()
            elif choice == "2":  #  Press 2 to add balance
                 self.deposit()
            elif choice == "3":  #  Press 3 to withdraw
                self.withdraw()
            elif choice == "4": #  Press 4 to check balance
                self.check_balance()
            elif choice == "5": # Press 5 to change pin
                self.change_pin()
            else:
                print("Thank you for using the ATM. Goodbye!")
                break 
        
        
    def create_pin(self):
        self.user_pin = input("Enter your pin: ")
        self.__pin = self.user_pin
        print("PIN created successfully!")
        
    def deposit(self):
        temp = input("Enter your Current pin to add Balance: ")
        if temp == self.__pin:
            amount = int(input("Enter the amount you want to add: "))
            self.__balance += amount
            print(f"{amount} is added to your account")
            print("Current Balance: ", self.__balance)
        else:
            print("Invalid Pin")
            
    def withdraw(self):
        temp = input("Enter your Current pin to add Withdraw: ")
        if temp == self.__pin:
            amount = int(input("Enter amount to withdraw: "))
            if amount <= self.__balance:
                self.__balance -= amount
                print(f"{amount} withdrawn successfully.")
            else:
                print("Insufficient balance!")
        else:
             print("Incorrect PIN!")
        
    
    def check_balance(self):
        temp = input("Enter your Current pin to check Balance: ")
        if temp == self.__pin:
            print("Your Current balance is:", self.__balance)
        else:
            print("Invalid Pin")
    
    def change_pin(self):
        old_pin = input("Enter your current pin: ")
        if old_pin == self.__pin:
            self.__pin = input("Enter your new pin: ")
            print("PIN changed successfully!")
        else:
            print("Incorrect PIN!")
            self.change_pin()
        
   
    
# hdfc = AtmMachine()
sbi = AtmMachine()  


Thank you for using the ATM. Goodbye!


**Bug fixed:** the printed menu text still described the *old* option numbering from the Section 1 ATM ("Press 0 to add balance", "Press 1 to create pin", ...), but the actual `if/elif` checks in this class use a completely different mapping (`"1"` = create pin, `"2"` = deposit, etc.). The menu text has been corrected to match what the code actually does — this was a real "the comment lies about the code" bug that would have confused anyone using it.

**Note:** `self.__menu` (with double underscore) is itself name-mangled too — it becomes `self._AtmMachine__menu` internally. This is why the interactive demo below still works fine from *inside* the class, but you could not call `sbi.__menu()` from outside without also mangling the name yourself.

**Gotcha:** `self.user_pin = input(...)` inside `create_pin` creates a *new, non-mangled* public attribute (`user_pin`) as well as setting `self.__pin`. That's an accidental leftover — it means the pin is technically still readable from outside via `sbi.user_pin`, defeating some of the purpose of using `__pin` in the first place. A stricter version would just do `self.__pin = input("Enter your pin: ")` directly.

<a id="single-underscore"></a>
### 3.1 The "Single Underscore" Convention (`_variable`)

A single leading underscore is a **convention only** — it signals "internal use, please don't touch" but Python does nothing to stop you.

In [6]:
class BankAccount:
    def __init__(self):
        self._balance = 1000  # Intended to be private
        
account = BankAccount()
print(account._balance)  # Works perfectly fine! 1000 | It acts like a "Keep Out" sign on an unlocked door. You can walk right in if you choose to ignore the sign.
        

1000


**Note:** `account._balance` works with zero errors — Python enforces *nothing* here. The underscore is purely a social contract between developers, like a "Keep Out" sign on an unlocked door.

<a id="name-mangling"></a>
### 3.2 Name Mangling (`__variable`)

A double leading underscore does something real: Python **renames** the attribute internally to `_ClassName__attribute`. This is called **name mangling**.

In [7]:
class BankAccount:
    def __init__(self):
        self.__vault_code = 9999  # Sounds private, right?

account = BankAccount()

print(account.__vault_code) 

# Accessing the mangled name directly:
# print(account._BankAccount__vault_code)  # Works perfectly! 9999

AttributeError: 'BankAccount' object has no attribute '__vault_code'

**Note:** This cell **intentionally errors** — `account.__vault_code` fails with `AttributeError` because Python silently renamed the attribute to `_BankAccount__vault_code` behind the scenes. There is no attribute literally called `__vault_code` on the object; name mangling happened at class-definition time.

In [ ]:
class BankAccount:
    def __init__(self):
        self.__vault_code = 9999  # Sounds private, right?

account = BankAccount()

# Accessing the mangled name directly:
print(account._BankAccount__vault_code)  # Works perfectly! 9999

9999


**Note:** Accessing the mangled name directly (`account._BankAccount__vault_code`) works and prints `9999`. This proves name mangling is **obfuscation, not real security** — the data is still there and still reachable if you know (or guess) the mangled name.

<a id="reflection"></a>
### 3.3 Full Reflection and Introspection

Using built-in functions like `dir()`, `vars()`, or `getattr()`, you can peek inside *any* object — including its mangled attributes — and read or modify them dynamically.

In [ ]:
class BankAccount:
    def __init__(self):
        self.owner = "Alice"
        self._balance = 1000
        self.__vault_code = 9999

account = BankAccount()

# Print the dictionary of attributes
print(vars(account))

# Pass the object inside dir()
print(dir(account))

# 1. Accessing a standard variable
print(getattr(account, "owner"))          # Output: Alice

# 2. Accessing a semi-private variable
print(getattr(account, "_balance"))       # Output: 1000

# 3. Accessing a mangled private variable
print(getattr(account, "_BankAccount__vault_code"))  # Output: 9999

# 4. Using a default value to prevent crashing if the attribute doesn't exist
print(getattr(account, "routing_number", "Not Found")) # Output: Not Found

{'owner': 'Alice', '_balance': 1000, '_BankAccount__vault_code': 9999}
['_BankAccount__vault_code', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_balance', 'owner']
Alice
1000
9999
Not Found


**Note:** `vars(account)` returns the object's actual `__dict__` — notice the key is `_BankAccount__vault_code`, not `__vault_code`, confirming the mangling happened at assignment time (`self.__vault_code = 9999` inside `__init__`), not just at read time.

**Note:** `getattr(obj, name, default)` is the safe way to read a possibly-missing attribute without a `try/except` — it returns `"Not Found"` here instead of raising `AttributeError` for `routing_number`, which was never set.

<a id="getter-setter"></a>
## 4. 🔑 Getter and Setter in Python

Since Python doesn't enforce privacy, getters/setters exist to add **validation** when reading or writing a value — not to "protect" it from access.

In [ ]:
# 1. The Traditional Way (Methods/Functions): If you prefer explicit function calls (like get_x() and set_x()), you can write standard methods.

class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    # Getter Method
    def get_balance(self):
        return self._balance

    # Setter Method
    def set_balance(self, new_amount):
        if new_amount >= 0:
            self._balance = new_amount
        else:
            print("Invalid amount")

# --- How to use it ---
account = BankAccount("Bob", 2000)

# Using the Getter method
print(account.get_balance())  # Output: 2000

# Using the Setter method
account.set_balance(2500)
print(account.get_balance())  # Output: 2500

2000
2500


**Note:** This is the traditional, Java-style approach: explicit `get_balance()` / `set_balance()` method calls. It works, but it's not the idiomatic way to do this in Python.

In [ ]:
# 2. The Pythonic Way (Recommended: @property)

class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance  # Internal variable

    # The Getter
    @property
    def balance(self):
        print("Fetching the balance safely...")
        return self._balance

   # The Setter
    @balance.setter
    def balance(self, new_amount):
        if new_amount < 0:
            print("Error: Balance cannot be negative!")
        else:
            print("Updating balance...") 
            self._balance = new_amount

# --- How to use it ---
account = BankAccount("Alice", 1000)

# Using the Getter (Notice: No parentheses needed!)
print(account.balance)      # Output: Fetching the balance safely... -> 1000

# Using the Setter (Notice: You use the assignment '=' operator)
account.balance = 1500      # Output: Updating balance...
account.balance = -500      # Output: Error: Balance cannot be negative!

print(account.balance)      # Latets Balance

Fetching the balance safely...
1000
Updating balance...
Error: Balance cannot be negative!
Fetching the balance safely...
1500


**Note:** With `@property`, `account.balance` (no parentheses!) calls the getter method behind the scenes, and `account.balance = 1500` (plain assignment) calls the setter — but the setter's validation (`if new_amount < 0`) still runs. This is the **Pythonic** way to add validation while keeping the clean "attribute access" syntax.

**Gotcha:** Once you define `balance` as a `@property`, you can no longer do `self.balance = ...` inside `__init__` without going through the setter too — that's actually a *feature* here (the setter's validation applies even during construction), but it can be a surprise if you expected `__init__` to bypass validation.

<a id="reference-variables"></a>
## 5. 🔗 Reference Variables in Python

When you pass an object to a function, you're passing a **reference** (the memory address), not a copy. This means the function can mutate the original object.

In [ ]:
class BankAccount:
    def __init__(self, name):
        self.name = name

def greet(customer_obj):
    print(id(customer_obj))

cust = BankAccount("Ankita")
print(id(cust))
greet(cust)

4475465168
4475465168


**Note:** `id(cust)` printed twice (once outside, once inside `greet`) shows the **same** number — `cust` and `customer_obj` are two different names pointing at the *same* object in memory.

In [ ]:
class BankAccount:
    def __init__(self, name):
        self.name = name

def greet(customer_obj):
    customer_obj.name = "Nitish"
    print(customer_obj.name)

cust = BankAccount("Ankita")
greet(cust)

print(cust.name)

Nitish
Nitish


**Note:** `greet` sets `customer_obj.name = "Nitish"` — this **mutates the object's attribute**, not the reference itself. Since `cust` and `customer_obj` point to the same object, `cust.name` is now `"Nitish"` too, even after the function returns.

In [ ]:
class BankAccount:
    def __init__(self, name):
        self.name = name

def greet(customer_obj):
    print(id(customer_obj))
    customer_obj.name = "Nitish"
    print(customer_obj.name)
    print(id(customer_obj))

cust = BankAccount("Ankita")
greet(cust)

print(cust.name)

## class ke object are also mutable like list dictionary and sets



4477005584
Nitish
4477005584
Nitish


**Note:** This combines the two ideas above — the `id()` stays identical before and after the mutation (same object, same address), but `.name` has changed. Mutating an attribute never changes an object's identity.

In [ ]:
class BankAccount:
    def __init__(self, name, gender):
        self.name = name
        self.gender = gender

def greet(customer_obj):
    if customer_obj.gender == "Male":
        print("Hello,", customer_obj.name, "Sir")
    elif customer_obj.gender == "Female":
        print("Hello,", customer_obj.name, "Mam")
    else:
         print("Wrong Gender Input")
         
    cust2 = BankAccount("Nitish", "Male")
    return cust2

cust = BankAccount("Ankita", "Female")
new_cust = greet(cust)
print(new_cust.name)

Hello, Ankita Mam
Nitish


**Note:** This one is a common source of confusion — inside `greet`, `customer_obj` refers to `cust` ("Ankita"), and the printed greeting correctly uses her data. But then the function creates a **brand-new** `BankAccount` object (`cust2`, "Nitish") and returns *that* instead of `cust`. So `new_cust.name` is `"Nitish"`, not `"Ankita"` — the original `cust` object was never modified; a completely different object was returned. This demonstrates that **reassigning inside a function (or returning a new object) does not affect the caller's original variable**, unlike mutating an existing object's attribute (as in the two cells above).

**Note:** Just like lists, dicts, and sets, class objects in Python are **mutable** — the two cells above are proof of this.

<a id="mutable-vs-immutable"></a>
### 5.1 Comparing to Lists and Tuples (Mutable vs Immutable)

Class objects behave like lists (mutable, mutation is visible to the caller) — but not like tuples (immutable, so "changing" one actually creates a new object).

In [ ]:
## List example of pass by reference

def change(L):
    print(id(L))
    L.append(5)
    print(id(L))
    

L1 = [1, 2, 3, 4]
print(id(L1))
print(L1)

change(L1)
print(L1)

print("---------")
L2 = [1, 2, 3, 4]
print(id(L2))
print(L2)

change(L2[:]) ## try to use clonning to avoid data rewrite in inside of method in python
print(L2)




4476070016
[1, 2, 3, 4]
4476070016
4476070016
[1, 2, 3, 4, 5]
---------
4476064704
[1, 2, 3, 4]
4475679680
4475679680
[1, 2, 3, 4]


**Note:** `change(L1)` mutates the list **in place** (`L.append(5)`) — the `id()` doesn't change, and `L1` outside the function now includes the `5` too. But `change(L2[:])` passes a **slice-copy** of `L2` — a brand-new list object — so appending to it inside the function has zero effect on the original `L2`. Slicing (`L2[:]`) is a classic trick to avoid unwanted mutation.

In [ ]:
## Tuple example of pass by reference

def change(L):
    print(id(L))
    L = L + (5, 6) ## here id will chnage but change will not happen in tupple
    print(id(L))
    

L1 = (1, 2, 3, 4)
print(id(L1))
print(L1)

change(L1)
print(L1)

4468905744
(1, 2, 3, 4)
4468905744
4476232672
(1, 2, 3, 4)


**Bug fixed (comment typos):** the original comment said "here id will chnage but change will not happen in tupple" — fixed the typos ("chnage" → "change", "tupple" → "tuple") for readability. The behavior itself was already correct: `L = L + (5, 6)` doesn't mutate the tuple — it creates an **entirely new tuple** and rebinds the *local* name `L` to it, which is why `id(L)` changes inside the function, but the original `L1` outside is untouched. This is the core reason tuples are immutable: there is no in-place "append" operation for them at all.

<a id="collections-of-objects"></a>
## 6. 📦 Collections of Objects

Objects can be stored in ordinary Python collections (lists, dicts, etc.) just like any other value.

In [ ]:
class Customer:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def intro(self):
        print("I am", self.name,"and my Age is", self.age)

c1 = Customer("Pratham", 23)
c2 = Customer("Isha", 54)
c3 = Customer("Anshu", 73)


L = [c1, c2, c3]

for i in L:
    print(i)

print("---------")

for i in L:
    print(i.name, i.age)
    
print("---------")

for i in L:
    i.intro()


---------
Pratham 23
Isha 54
Anshu 73
---------
I am Pratham and my Age is 23
I am Isha and my Age is 54
I am Anshu and my Age is 73


**Note:** `print(i)` on a raw object shows Python's default representation (`<__main__.Customer object at 0x...>`) because `Customer` doesn't define `__str__` — compare this to the `Fraction` class in Section 2, which *does* define `__str__` and therefore prints nicely.

<a id="static-vs-instance"></a>
## 7. 🧮 Static (Class) Variables vs Instance Variables

A **class variable** is shared by every object of the class — perfect for things like counters. An **instance variable** belongs to just one object.

In [ ]:
## Static Variable and Instance Variable

class Customer:
    
    counter = 1
    
    def __init__(self, name, age):
        self.name = name  ## instance variable
        self.age = age ## instance variable
        self.sno = 0
        self.sno += 1
        # print(id(self.sno))
        self.counter = Customer.counter
        Customer.counter += 1
    
    
    def intro(self):
        print("I am", self.name,"and my Age is", self.age)

c1 = Customer("Pratham", 23)
c2 = Customer("Isha", 54)
c3 = Customer("Anshu", 73)

print(c1.sno) ## non static variable
print(c1.sno)
print(c1.sno)
print("----------")
print(c1.counter) ## static variable
print(c2.counter)
print(c3.counter)

1
1
1
----------
1
2
3


**Note:** `self.sno = 0` followed immediately by `self.sno += 1` always resets `sno` to `0` and then increments it to `1` for *every* object — so `c1.sno`, `c2.sno`, and `c3.sno` are all `1`. This variable isn't actually doing what its name ("serial number") suggests; it never persists or increments across objects because it's reset inside `__init__` every time. Compare this to `counter`, which correctly increases (`1`, `2`, `3`) because it reads and writes the shared **class** attribute `Customer.counter` instead of resetting a fresh instance value each time.

<a id="private-static-counter"></a>
### 7.1 A Private Static Counter with a Static Method

In [ ]:
## Static Variable and Instance Variable

class Atm:
    
    __counter = 1
    
    def __init__(self, name, age):
        self.name = name  ## instance variable
        self.age = age ## instance variable
        self.__counter = Atm.__counter
        Atm.__counter += 1
       
    @staticmethod
    def get_counter(): ## static method
        return Atm.__counter

    def set_counter(new):
        if type(new) is int:
            Atm.__counter = new
        else:
            print("Not Allowed")
        
    
    def intro(self):
        print("I am", self.name,"and my Age is", self.age)

# c1 = Atm("Pratham", 23)
# c2 = Atm("Isha", 54)

# print(Atm.get_counter())
# c3 = Atm("Anshu", 73)


print(Atm.get_counter())

Atm.set_counter(5)
print(Atm.get_counter())


# print(c1.__counter) ## private static variable cant be accessible use not getter and setter methods
# print(c2.__counter)
# print(c3.__counter)



1
5


**Bug found:** `set_counter(new)` is missing the `@staticmethod` decorator. As written, calling it the way a static method is meant to be called — e.g. `Atm.set_counter(5)` — happens to work only because Python doesn't require an instance when you call a method **through the class itself** (the `new` parameter absorbs the `5`). But calling it on an *instance* (e.g. `some_atm.set_counter(5)`) would crash with `TypeError: set_counter() takes 1 positional argument but 2 were given`, because Python would automatically pass the instance as the first argument, leaving no slot for `5`. Verified below — adding `@staticmethod` fixes it so it works consistently either way.

In [ ]:
## Demonstrating the missing @staticmethod bug, and the fix

class AtmBuggy:
    __counter = 1
    def set_counter(new):   # missing @staticmethod
        Atm_class_ref = new
        return Atm_class_ref

class AtmFixed:
    __counter = 1
    @staticmethod
    def set_counter(new):
        AtmFixed.__counter = new
        return AtmFixed.__counter

buggy = AtmBuggy()
try:
    buggy.set_counter(5)   # crashes: instance auto-passes 'self', leaving no room for 5
except TypeError as e:
    print("AtmBuggy.set_counter on an instance ->", e)

fixed = AtmFixed()
print("AtmFixed.set_counter on an instance ->", fixed.set_counter(5))  # works fine


**Note:** This confirms the fix — with `@staticmethod` added, `set_counter` can be called consistently through the class (`Atm.set_counter(5)`) *or* through an instance (`some_atm.set_counter(5)`) without Python trying to auto-inject `self`.

<a id="aggregation"></a>
## 8. 🧩 Class Relationships — Aggregation ("Has-A")

Aggregation means one class holds an object of another class as an attribute. Here, a `Customer` **has an** `Address`.

In [ ]:
class Customer:
    
    def __init__(self, name, gender, address):
        self.name = name
        self.gender = gender
        self.address = address
    
    def edit_profile(self,new_name, new_city, new_pin, new_state):
        self.name = new_name
        self.address.change_address(new_city, new_pin, new_state)

class Address:
    
    def __init__(self, city, pincode, state):
        self.city = city
        self.pincode = pincode
        self.state = state
        
    def change_address(self, new_city, new_pin, new_state):
        self.city = new_city
        self.pincode = new_pin
        self.state = new_state
        
        
add = Address("Bengaluru",829122, "KA")
cust = Customer("Pratham", "Male", add)

print(cust.address)
print(cust.address.city)

cust.edit_profile("Ankit","Gurgano", 12001, "HR")

print(cust.address)
print(cust.address.city)        

Bengaluru
Gurgano


**Note:** `cust.edit_profile(...)` doesn't touch `self.address` directly — it delegates to `self.address.change_address(...)`, letting the `Address` object manage its own fields. This is the payoff of aggregation: each class stays responsible for its own data.

**Note:** `print(cust.address)` shows the default object representation (`<__main__.Address object at 0x...>`) both before and after editing, and the `id()` doesn't change — because `edit_profile` mutates the *existing* `Address` object's attributes rather than replacing it with a new one.

<a id="inheritance"></a>
## 9. 🌳 Class Relationships — Inheritance ("Is-A")

Inheritance means a subclass automatically gets all the methods (and attributes) of its parent class. Here, a `Student` **is a** `User`.

**Real-life analogy:** Think of a **family recipe book** 👩‍👧. A child inherits their parent's base recipes (methods) automatically — they don't need to rewrite the "how to boil rice" recipe from scratch. They can use it as-is, or tweak it slightly (**override** it) to add their own twist, while still being fundamentally the same "cook" their parent was.

**Why we need it:** without inheritance, every related class would have to duplicate the same methods over and over. `Student`, `Teacher`, and `Admin` might all need `login()` and `register()` — inheritance lets you write that logic **once**, in a shared parent (`User`), and reuse it everywhere.

### 🆚 How Python Treats Inheritance Differently

| | Java / C# | C++ | Python |
|---|---|---|---|
| Multiple inheritance (one class, several parents) | ❌ Not allowed for classes — only via `interface` | ✅ Allowed, but no built-in conflict resolution — the programmer must resolve ambiguity manually | ✅ Allowed natively — `class Child(Father, Mother):` |
| Resolving conflicts (same method name in two parents) | N/A (can't happen with single class inheritance) | Ambiguous — compiler error unless manually qualified (`Father::skill()`) | Automatic — Python uses a defined algorithm called **MRO** (Method Resolution Order) |
| "Is-A" enforcement | Compiler-checked at every call | Compiler-checked at every call | Not enforced at all — duck typing means Python only checks that the method exists when it's actually called |

> 🧸 Python is one of the few mainstream languages that allows **true multiple inheritance** for regular classes, and it doesn't leave conflict resolution up to guesswork — every Python class has a well-defined, computable **Method Resolution Order** (see 9.3 below) that decides exactly which parent's method wins.

<a id="inheritance-types"></a>
### 9.1 Types of Inheritance

Python supports four shapes of inheritance:

| Type | Shape | Meaning |
|---|---|---|
| **Single** | `B(A)` | One child, one parent |
| **Multilevel** | `C(B)`, `B(A)` | A chain — grandchild inherits from child, which inherits from grandparent |
| **Hierarchical** | `B(A)`, `C(A)` | Multiple children share the SAME parent |
| **Multiple** | `C(A, B)` | One child inherits from TWO (or more) parents at once |

#### Single Inheritance — One Parent, One Child

This is the simplest case: `Student` inherits from exactly one parent, `User`.

In [ ]:
class User:
    
    def login(self):
        print("Login")

    def register(self):
        print("register")
    
class Student(User):
    
    def enroll(self):
        print("Enroll")
        
    def review(self):
        print("Review")
    
stu1 = Student()

stu1.enroll()
stu1.review()
stu1.login()
stu1.register()


Enroll
Review
Login
register


**Note:** `Student` doesn't define `login()` or `register()` itself — it inherits them from `User` automatically just by writing `class Student(User):`. This is the core benefit of inheritance: shared behavior is written once in the parent and reused everywhere. This is **Single Inheritance** — one child, one parent.

#### Multilevel Inheritance — A Chain of Generations

Here, `Puppy` inherits from `Dog`, which inherits from `Animal` — so `Puppy` ends up with abilities from BOTH ancestors, not just its direct parent.

In [ ]:
class Animal:
    def eat(self):
        print("Eating...")

class Dog(Animal):        # Dog is-a Animal
    def bark(self):
        print("Barking...")

class Puppy(Dog):          # Puppy is-a Dog, which is-a Animal
    def weep(self):
        print("Weeping...")

p = Puppy()
p.eat()     # inherited from Animal (grandparent)
p.bark()    # inherited from Dog (parent)
p.weep()    # defined directly on Puppy


**Note:** `Puppy` never mentions `Animal` at all — `eat()` reaches it through the chain `Puppy → Dog → Animal`. This is why it's called *multilevel*: the inheritance passes down through more than one level, like a grandchild inheriting a trait from a grandparent even though their parent is the one standing in between.

#### Hierarchical Inheritance — One Parent, Many Children

Here, both `Car` and `Bike` inherit from the SAME parent, `Vehicle` — the opposite shape of multilevel inheritance.

In [ ]:
class Vehicle:
    def start(self):
        print("Vehicle started")

class Car(Vehicle):        # Car is-a Vehicle
    def wheels(self):
        print("4 wheels")

class Bike(Vehicle):        # Bike is-a Vehicle too -- SAME parent as Car
    def wheels(self):
        print("2 wheels")

car = Car()
car.start()    # inherited from Vehicle
car.wheels()    # Car's own version

bike = Bike()
bike.start()    # the SAME inherited method, reused by a totally different sibling class
bike.wheels()


**Note:** `Car` and `Bike` each override `wheels()` with their own version (this is **method overriding** — see 9.2 below), but neither needs to redefine `start()` — both get it for free from `Vehicle`. This avoids duplicating `start()` in every vehicle subclass.

#### Multiple Inheritance — One Child, Two (or More) Parents

`Child` inherits from BOTH `Father` AND `Mother` at once — something Java and C# don't allow for regular classes at all.

In [ ]:
class Father:
    def skills(self):
        print("Gardening, Programming")

class Mother:
    def skills(self):
        print("Cooking, Art")

class Child(Father, Mother):    # inherits from BOTH parents
    pass

c = Child()
c.skills()   # whose skills() wins -- Father's or Mother's? See the note below.


**Note:** `Child.skills()` printed `"Gardening, Programming"` — **Father's** version won, not Mother's. Both parents define `skills()`, so which one runs isn't arbitrary — it's decided by a precise rule called MRO. Let's look at that rule directly.

<a id="mro"></a>
### 9.2 Method Resolution Order (MRO) and the Diamond Problem

When a class has multiple parents that define the SAME method, Python needs a deterministic rule to decide which one wins. That rule is the **Method Resolution Order (MRO)** — the exact left-to-right, depth-first order Python searches through parent classes.

> 🧸 Think of MRO like a **line of succession** for a throne. When `Child(Father, Mother)` is written, Python doesn't guess — it lays out an exact search order: `Child` first, then `Father` (the first-listed parent), then `Mother`, then `object` (every class's ultimate ancestor). The first matching method found along that line wins.

The classic **"diamond problem"** — where two parents share a common grandparent, forming a diamond shape — is exactly what MRO exists to resolve cleanly, without the ambiguity C++ leaves to the programmer.

In [ ]:
print(Child.mro())
# or equivalently:
print(Child.__mro__)


**Note:** The output — `[Child, Father, Mother, object]` — is the exact search path. Python looked for `skills()` on `Child` first (not found), then `Father` (found it! stop here) — `Mother`'s version is never even reached for this particular call. This is why the earlier example printed Father's skills, not Mother's: **the order you list parents in `class Child(Father, Mother):` directly decides the MRO**. Writing `class Child(Mother, Father):` instead would flip the result.

> ❌ Misconception: with multiple inheritance, if two parents define the same method, Python raises an error or picks randomly.
> ✅ Correct: Python always resolves it deterministically via MRO — leftmost-listed parent wins ties, computed using an algorithm called C3 linearization. Every class's MRO can be inspected directly with `.mro()`.

<a id="super-overriding"></a>
### 9.3 `super()` and Method Overriding

**Method overriding** means a subclass defines a method with the SAME name as one in its parent, replacing the parent's behavior for that subclass. `super()` lets the child still call the parent's original version *from inside* the override — extending it instead of fully replacing it.

In [10]:
class Animal:
    def speak(self):
        print("Animal is speaking")

    def display(self):
        print("This is display function")

class Dog(Animal):
    def speak(self):
        super().speak()
        print("Dog is Speaking")
        self.display()
        print("Dog is Barking")


dd = Dog()
dd.speak()


Animal is speaking
Dog is Speaking
This is display function
Dog is Barking


In [ ]:
## Mutliple Inheritance

class Flyer:
    def fly(self):
        print("Flying")

class Swimmer:
    def swim(self):
        print("Swimming")

class Duck(Flyer, Swimmer):
    def quack(self):
        print("Quacks!")


dd = Duck()
dd.fly()
dd.swim()
dd.quack()

Flying
Swimming
Quacks!


In [8]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def display(self):
        print(f"Name: {self.name}, Salary: {self.salary}")

class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)   # calls Employee's __init__ to set name/salary
        self.team_size = team_size        # then adds Manager's own extra attribute

    def display(self):                     # OVERRIDES Employee's display()
        super().display()                   # ...but still calls the parent's original version first
        print(f"Team Size: {self.team_size}")   # ...then extends it with more info

m = Manager("Priya", 90000, 8)
m.display()


Name: Priya, Salary: 90000
Team Size: 8


**Note:** `Manager.__init__` doesn't repeat `self.name = name; self.salary = salary` — it calls `super().__init__(name, salary)` and lets `Employee` do that work once. Same idea in `display()`: `super().display()` runs the ORIGINAL parent version first, and then the override adds its own extra line, rather than throwing the parent's logic away entirely. This is the difference between overriding that **replaces** behavior (a plain `def display(self):` with no `super()` call) and overriding that **extends** it (calling `super()` first).

**A simpler override — full replacement, no `super()`:**

In [ ]:
class Bird:
    def sound(self):
        print("Some generic bird sound")

class Parrot(Bird):
    def sound(self):                              # completely REPLACES Bird's sound(), no super() call
        print("Parrot says: I'm a good boy!")

pr = Parrot()
pr.sound()   # only Parrot's version runs -- Bird's version is fully shadowed


**Note:** `Parrot.sound()` has no `super().sound()` call — `Bird`'s generic version is completely shadowed. Compare to `Manager.display()` above, which chose to call `super()` and build on top of the parent instead.

<a id="isinstance-issubclass"></a>
### 9.4 `isinstance()` and `issubclass()`

These two answer different questions: `isinstance(obj, Class)` asks **"is this OBJECT an instance of this class (or one of its ancestors)?"** `issubclass(ClassA, ClassB)` asks **"does this CLASS inherit from that class?"** — no object involved at all, just the class relationship itself.

In [ ]:
m = Manager("Priya", 90000, 8)   # from the super() example above

print(isinstance(m, Manager))    # True  -- m is directly a Manager
print(isinstance(m, Employee))   # True  -- m is ALSO an Employee, because Manager inherits from it
print(isinstance(m, object))      # True  -- every class ultimately inherits from object

print(issubclass(Manager, Employee))   # True  -- Manager DOES inherit from Employee
print(issubclass(Employee, Manager))   # False -- Employee does NOT inherit from Manager (wrong direction!)


**Note:** `isinstance(m, Employee)` is `True` even though `m` was created as a `Manager` — this is exactly what makes inheritance an "Is-A" relationship: a `Manager` genuinely **is an** `Employee` too, not just something *similar* to one. `issubclass()` checks the class hierarchy directly, without needing any object at all — this is why `issubclass(Employee, Manager)` is `False`: the relationship only goes one direction, from child to ancestor, never the other way around.

<a id="polymorphism"></a>
## 10. 🎭 Polymorphism ("Many Forms")

**Polymorphism** means the same method call — `obj.sound()`, `obj.area()` — behaves differently depending on WHICH object it's actually called on. The word literally means "many forms": one interface, many implementations.

**Real-life analogy:** Think of a **"Play" button** ▶️ on any media app. Pressing "Play" on a song plays audio. Pressing "Play" on a video plays video. Pressing "Play" on a podcast starts speech. Same button, same action name — completely different behavior depending on what you're playing.

**Why we need it:** without polymorphism, you'd need a different function name for every type — `play_song()`, `play_video()`, `play_podcast()` — and every place that uses them would need to know which one to call. Polymorphism lets you write ONE piece of code (`make_sound(obj)`, a loop over `shapes`) that works correctly no matter what kind of object it receives, as long as that object supports the expected method.

### 🆚 How Python Treats Polymorphism Differently

| | Java / C# | Python |
|---|---|---|
| Requires a shared parent/interface? | Usually yes — the compiler checks that the object's declared type has the method | No — Python only cares that the object HAS the method **at the moment it's called**, regardless of its type or ancestry |
| Method overloading (same name, different parameters) | Built-in language feature — the compiler picks the right version based on argument types/count | Not supported the same way — defining a method twice just **replaces** the first version (see 10.2 below); default/variable arguments are used instead |
| Checking timing | Compile time | Runtime — Python simply tries to call the method, and raises `AttributeError` if it doesn't exist |

> 🧸 This runtime-only checking is **duck typing** again (first introduced in [Part 1's Abstraction section](15_Oops_part1.ipynb)): *"If it walks like a duck and quacks like a duck, treat it like a duck."* Python doesn't check whether `Duck`, `Dog`, and `Car` share a common parent before calling `.sound()` on each — it just tries the call, and it works as long as the method exists.

### 10.1 Duck Typing — Polymorphism Without Inheritance

Here, `Duck`, `Dog`, and `Car` share **no common parent at all** — not even a hint of a relationship — yet the exact same function works on all three.

In [ ]:
class Duck:
    def sound(self):
        print("Quack quack")

class Dog:
    def sound(self):
        print("Woof woof")

class Car:                  # totally unrelated to Duck/Dog -- not even an animal!
    def sound(self):
        print("Vroom vroom")

def make_sound(obj):
    obj.sound()             # doesn't check obj's type at all -- just calls .sound() and hopes it exists

for thing in [Duck(), Dog(), Car()]:
    make_sound(thing)


**Note:** `make_sound()` never checks `type(obj)` or requires a shared parent class — it just calls `obj.sound()` and trusts that it exists. This is duck typing in action: Python is happy to treat `Duck`, `Dog`, and `Car` interchangeably, purely because they all happen to define a `.sound()` method — even though `Car` isn't remotely an animal.

<a id="method-overloading"></a>
### 10.2 Method Overloading — Why Python Doesn't Have It (the Java Way)

**Method overloading** (in Java/C++) means defining the SAME method name multiple times with different parameter lists, and the compiler picks the right version based on how many arguments you pass. Python has no such feature — defining a method twice in the same class just **silently replaces** the first definition with the second.

In [ ]:
class Calculator:
    def add(self, a, b):
        return a + b

    def add(self, a, b, c):   # this SECOND definition completely REPLACES the first -- not an overload!
        return a + b + c

calc = Calculator()

try:
    print(calc.add(2, 3))       # fails -- the 2-argument version doesn't exist anymore
except TypeError as e:
    print("TypeError:", e)

print(calc.add(2, 3, 4))          # only the 3-argument version survives


> ❌ Misconception: writing `def add(self, a, b):` and then `def add(self, a, b, c):` in the same class overloads `add()`, like in Java.
> ✅ Correct: Python classes only ever keep the LAST definition of a method with a given name — the first one is simply gone. Python achieves what overloading is *used for* through **default arguments** or `*args`, shown below.

**The Pythonic fix — default arguments:**

In [ ]:
class CalculatorFixed:
    def add(self, a, b, c=0):   # c defaults to 0, so it becomes optional
        return a + b + c

calc2 = CalculatorFixed()
print(calc2.add(2, 3))        # works with 2 args -- c defaults to 0
print(calc2.add(2, 3, 4))     # also works with 3 args


**An even more flexible fix — `*args` (any number of arguments):**

In [ ]:
class CalculatorFlexible:
    def add(self, *args):        # accepts ANY number of arguments, collected into a tuple
        return sum(args)

calc3 = CalculatorFlexible()
print(calc3.add(2, 3))              # 2 args
print(calc3.add(2, 3, 4))            # 3 args
print(calc3.add(1, 2, 3, 4, 5))       # 5 args -- still works, no new method needed


**Note:** `*args` collects every positional argument into a tuple, so `add()` now works with 2, 3, or 5 numbers without ever needing a second definition. This is the idiomatic Python replacement for "I want this method to accept a variable number of arguments," which is one of the main reasons Java/C++ reach for overloading in the first place.

<a id="overriding-as-polymorphism"></a>
### 10.3 Method Overriding as Polymorphism (via Inheritance)

This connects directly back to [Section 9.3](#super-overriding) — when subclasses **override** a shared parent method with their own version, calling that SAME method name on different objects produces different results. This is polymorphism achieved *through* inheritance, rather than through duck typing alone.

In [ ]:
class Shape:
    def area(self):
        pass   # no implementation -- just a placeholder, like the abstract methods from Part 1

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return round(3.14159 * self.radius ** 2, 2)

class Rectangle(Shape):
    def __init__(self, length, width):
        self.length = length
        self.width = width
    def area(self):
        return self.length * self.width

shapes = [Circle(5), Rectangle(4, 6)]

for shape in shapes:
    print(f"{type(shape).__name__} area: {shape.area()}")   # SAME method call, DIFFERENT calculation each time


**Note:** The `for` loop calls `shape.area()` identically for every item, with no `if isinstance(shape, Circle): ... elif isinstance(shape, Rectangle): ...` branching needed. Each object already knows HOW to calculate its own area — the loop just asks, and polymorphism ensures the right version runs. This is the real payoff of combining inheritance with overriding: code that scales to new shapes (`Triangle`, `Hexagon`, ...) without ever needing to change the loop itself, as long as each new class also defines `area()`.

<a id="dunder-methods"></a>
## 11. 🪄 Dunder Methods ("Magic Methods")

**Dunder** is short for "**D**ouble **UNDER**score" — methods like `__init__`, `__str__`, and `__add__` that start and end with `__`. You've already used several throughout this notebook (`__init__` in every class, `__str__`/`__add__` in `Fraction` back in Section 2, `__mro__` in Section 9) without a dedicated explanation of what they actually are, as a category.

**What they really are:** dunder methods are Python's way of letting your own classes **plug into built-in syntax and functions** — `+`, `==`, `<`, `len()`, `print()`, `str()`, `[...]` indexing, and more. Python doesn't have special-case code for "how do I add two `Fraction` objects?" — it just looks for a `__add__` method on the object and calls it. This is sometimes called **operator overloading**, but it's really just one application of a much bigger, more general mechanism.

**Real-life analogy:** Think of dunder methods like **standardized plug sockets** 🔌. Any appliance that has the right plug shape (implements the right dunder method) can be plugged into the same wall socket (`+`, `print()`, `len()`, ...) — the socket doesn't care what the appliance actually is, only that it fits the expected interface.

### 🆚 How Python Treats This Differently

| | Java / C++ | Python |
|---|---|---|
| Operator overloading | C++ allows it via `operator+()` syntax; Java doesn't allow it at all (except for built-in `String +`) | Universal — nearly every operator and built-in function has a corresponding dunder method any class can implement |
| Customizing `print()` / string conversion | Java: override `toString()` | Python: define `__str__` (and/or `__repr__`) |
| Customizing equality (`==`) | Java: override `.equals()` | Python: define `__eq__` |

> 🧸 Where Java gives you a handful of specific override points (`toString()`, `equals()`, `hashCode()`), Python generalizes the idea into dozens of dunder hooks — covering everything from arithmetic to comparisons to iteration to indexing — all following the exact same `__name__` naming pattern.

### 11.1 Common Dunder Methods in Action

One class, several dunders, each plugging into a different piece of built-in syntax.

In [ ]:
class Point:
    def __init__(self, x, y):        # plugs into: Point(1, 2)
        self.x = x
        self.y = y

    def __str__(self):                 # plugs into: print(p), str(p)
        return f"Point({self.x}, {self.y})"

    def __repr__(self):                 # plugs into: repr(p), and how p shows up INSIDE a list/dict
        return f"Point(x={self.x}, y={self.y})"

    def __eq__(self, other):             # plugs into: p1 == p2
        return self.x == other.x and self.y == other.y

    def __lt__(self, other):              # plugs into: p1 < p2 (compares distance from origin here)
        return (self.x**2 + self.y**2) < (other.x**2 + other.y**2)

    def __len__(self):                     # plugs into: len(p)
        return int((self.x**2 + self.y**2) ** 0.5)


p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)

print(p1)              # uses __str__
print(repr(p1))          # uses __repr__
print(p1 == p2)           # uses __eq__ -- True, same coordinates
print(p1 == p3)            # uses __eq__ -- False
print(p1 < p3)               # uses __lt__ -- True, p1 is closer to the origin
print(len(p3))                 # uses __len__
print([p1, p3])                  # a list of Points uses __repr__ for EACH item, not __str__


**Note:** `print([p1, p3])` displayed `Point(x=1, y=2)` style output — that's `__repr__`, NOT `__str__`. This is a genuinely common gotcha:

> ❌ Misconception: `__str__` and `__repr__` are basically the same thing — define one, and you're covered everywhere.
> ✅ Correct: `print(obj)` and `str(obj)` use `__str__`. But containers (a `list`, `dict`, or the interactive console echoing a value) use `__repr__` for each item, NOT `__str__`. If you only define `__str__`, printing a single object looks fine, but printing a LIST of them falls back to Python's default `<...object at 0x...>` representation for each item, unless `__repr__` is defined too. Best practice: always define `__repr__`; `__str__` is optional if you want a friendlier version specifically for `print()`.

<a id="eq-hash-gotcha"></a>
### 11.2 The `__eq__` / `__hash__` Gotcha

Defining `__eq__` on a class has a side effect most beginners don't expect: it silently disables the ability to use that object in a `set` or as a `dict` key.

In [1]:
class BadPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __eq__(self, other):              # defining __eq__ alone...
        return self.x == other.x and self.y == other.y

p = BadPoint(1, 2)
try:
    s = {p}                                 # ...silently makes BadPoint UNHASHABLE
except TypeError as e:
    print("TypeError:", e)


class GoodPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    def __hash__(self):                      # explicitly restore hashability
        return hash((self.x, self.y))

g = GoodPoint(1, 2)
s2 = {g}                                        # works fine now
print(s2)


TypeError: unhashable type: 'BadPoint'
{<__main__.GoodPoint object at 0x10cb6f590>}


**Note:** By default, every Python object is hashable (usable in a `set`/as a `dict` key) using its identity. The moment you define `__eq__` yourself, Python reasons: "if you're changing what *equal* means, your old identity-based hash no longer makes sense" — and automatically sets `__hash__` to `None`, making the whole object unhashable, unless you explicitly define `__hash__` too (this connects directly back to [Part 1's hashability discussion](15_Oops_part1.ipynb) for tuples). The fix is simple: whenever you define `__eq__`, also define `__hash__` based on the same fields used for equality.

### 📚 Cheat Sheet — Common Dunder Methods

| Dunder | Triggered By | Purpose |
|---|---|---|
| `__init__` | `MyClass(...)` | Constructor — set up a new object |
| `__str__` | `print(obj)`, `str(obj)` | Human-readable string |
| `__repr__` | `repr(obj)`, console echo, inside containers | Developer-facing / unambiguous string |
| `__eq__` | `obj1 == obj2` | Custom equality check |
| `__lt__`, `__gt__`, etc. | `obj1 < obj2`, `obj1 > obj2` | Custom comparisons (also enables `sorted()`) |
| `__add__`, `__sub__`, `__mul__` | `obj1 + obj2`, etc. | Operator overloading (Section 2's `Fraction`) |
| `__len__` | `len(obj)` | Custom length |
| `__hash__` | `hash(obj)`, `set`/dict-key usage | Must be defined alongside `__eq__` to stay hashable |
| `__getitem__` | `obj[index]` | Custom indexing/slicing behavior |
| `__call__` | `obj()` | Makes an object itself callable like a function |

<a id="abstraction"></a>
## 12. 🎭 Abstraction — Formalizing the Interface

Part 1 introduced abstraction through informal examples (`Car`, `CoffeeMachine`) and one formal `abc` example (`PaymentMethod`). Now that you've seen Inheritance, Polymorphism, and Dunder methods, we can go deeper: abstraction isn't just "hide the details" — it's about defining a **contract** that every subclass must honor, and Python gives you two distinct tools for writing that contract precisely: the **Template Method pattern** (mixing concrete and abstract methods in one base class) and **abstract properties** (forcing a subclass to expose a value, not just a method).

**Real-life analogy:** Think of a **restaurant franchise's operating manual** 🍔. Head office defines a fixed process every branch MUST follow — "greet the customer, take the order, serve the food" (steps that never change) — but leaves ONE step blank for each city to fill in: "prepare the food" — a Tokyo branch fries tempura, a Texas branch grills brisket. The manual is a template with fixed steps AND a required, branch-specific step. That's exactly what an abstract base class with both concrete and abstract methods does in code.

### 🆚 How Python Treats This Differently (the Deeper Angle)

| | Java / C++ | Python |
|---|---|---|
| Mixing "already implemented" and "must implement" in one contract | An `abstract class` can have concrete methods AND `abstract` methods together — same as Python | `ABC` + `@abstractmethod` supports the exact same mix — a class can have any number of normal methods alongside abstract ones |
| Forcing a subclass to expose a VALUE (not just a method) | An `interface` can declare a method like `getSalary()`, but can't force a **field-like** access pattern | `@property` stacked with `@abstractmethod` forces subclasses to expose `obj.salary` (attribute syntax), not `obj.get_salary()` — the contract includes HOW it's accessed, not just that it exists |
| What happens if a class implements *some* but not all abstract members | Compile error — won't build until every abstract method is overridden | `TypeError` at the moment you try to `Instantiate()` — checked per-class, so even a class that implements 9 of 10 abstract methods still can't be created |

> 🧸 The Template Method pattern flips the usual "who calls whom" direction: normally YOU call the parent's methods from the child (via `super()`, [Section 9.3](#super-overriding)). Here, the PARENT calls the (not-yet-written) child method from inside its own concrete method — the parent defines the skeleton of an algorithm and delegates just one step to whichever subclass fills in the abstract piece.

<a id="abstraction-template-method"></a>
### 12.1 The Template Method Pattern — Fixed Steps, One Required Gap

`Notification` below defines TWO concrete methods (`log`, and `notify` which orchestrates the whole process) and ONE abstract method (`send`). Every subclass inherits the fixed process for free, and only needs to fill in `send()`.

In [ ]:
from abc import ABC, abstractmethod

class Notification(ABC):
    def log(self, message):           # CONCRETE -- shared by every subclass
        print(f"[LOG] Sending: {message}")

    @abstractmethod
    def send(self, message):           # ABSTRACT -- every subclass must implement
        pass

    def notify(self, message):          # CONCRETE -- calls the abstract method without knowing HOW it works
        self.log(message)
        self.send(message)

class EmailNotification(Notification):
    def send(self, message):
        print(f"Emailing: {message}")

class SMSNotification(Notification):
    def send(self, message):
        print(f"Texting: {message}")


email = EmailNotification()
email.notify("Your order has shipped!")

sms = SMSNotification()
sms.notify("Your OTP is 4321")

# The base class itself still can't be instantiated -- it has an unfilled abstract method
try:
    n = Notification()
except TypeError as e:
    print(f"TypeError: {e}")


**Note:** `notify()` is the "algorithm skeleton" — it's written ONCE, in the parent, and never repeated in `EmailNotification` or `SMSNotification`. Each subclass only supplies the ONE piece that genuinely differs between an email and a text message: HOW `send()` actually delivers the message. `log()` demonstrates that abstract base classes aren't required to be *purely* abstract — mixing concrete helper methods alongside abstract ones is completely normal and often the whole point. And just like every other `ABC` example so far, `Notification()` itself still can't be instantiated — the unfilled `send()` blocks it.

**Compare this to [Section 10.3](#overriding-as-polymorphism)'s `Shape`/`Circle`/`Rectangle`:** that was plain method overriding with NO `abc` involved — Python let `Shape()` itself be created just fine, and only a *convention* (an empty `pass` body) suggested it shouldn't be used directly. `Notification` is the stricter version: `ABC` + `@abstractmethod` make that "please don't instantiate the base" rule an enforced one, not just a hint.

<a id="abstraction-properties"></a>
### 12.2 Abstract Properties — Forcing a VALUE, Not Just a Method

Sometimes the contract you want to enforce isn't "must have this method," but "must expose this piece of data, accessed like an attribute." Stacking `@property` under `@abstractmethod` does exactly that: subclasses must implement it as a `@property` too, so it's read as `emp.salary`, never `emp.salary()`.

In [ ]:
from abc import ABC, abstractmethod

class Employee(ABC):
    @property
    @abstractmethod
    def salary(self):
        pass

class Developer(Employee):
    def __init__(self, base):
        self.base = base
    @property
    def salary(self):
        return round(self.base * 1.1, 2)   # 10% bonus

class Manager(Employee):
    def __init__(self, base):
        self.base = base
    @property
    def salary(self):
        return round(self.base * 1.2, 2)   # 20% bonus


d = Developer(50000)
m = Manager(50000)
print(d.salary)   # accessed like an attribute -- NOT d.salary()
print(m.salary)

# The base class still can't be instantiated -- salary is still unfilled at this level
try:
    e = Employee()
except TypeError as e:
    print(f"TypeError: {e}")

# A subclass that forgets to implement the abstract property is blocked too
class Intern(Employee):
    def __init__(self, base):
        self.base = base
    # forgot to implement the salary property!

try:
    i = Intern(20000)
except TypeError as e:
    print(f"TypeError: {e}")


**Note:** `d.salary` is read with plain attribute syntax, no parentheses — exactly like `d.salary()` would be if `salary` were a normal method, except it isn't one. That's the point of stacking `@property` under `@abstractmethod`: the *contract* itself specifies "expose this as a property," not just "have a method with this name." `Employee()` still fails for the same reason as every other `ABC` in this notebook — `salary` was never filled in at the base level. `Intern` shows the enforcement is real, not just for the base class: forgetting to implement `salary` (as a property, on the *subclass*) blocks `Intern` from being created too, with the exact same `TypeError` pattern as `Notification` and `Shape` before it.

**Gotcha (caught during verification):** `self.base * 1.1` without `round()` produces an ugly floating-point result like `55000.00000000001` instead of a clean `55000.0`, due to how binary floating-point represents `1.1`. Always `round()` money-like calculations to a fixed number of decimal places before displaying them.

### 📚 Cheat Sheet — Comparing All Three Ways to Abstract Something

| Approach | Enforcement | When to reach for it |
|---|---|---|
| Naming convention (`_method`, [Part 1's `CoffeeMachine`](15_Oops_part1.ipynb)) | None — purely social contract | Quick internal/external split, no strict requirement that anyone implement anything |
| `ABC` + `@abstractmethod` on a plain method ([`PaymentMethod`](15_Oops_part1.ipynb), `Notification` above) | Enforced at instantiation — `TypeError` if any abstract method is missing | You need every subclass to implement a specific BEHAVIOR, and want Python to actively block incomplete subclasses |
| `ABC` + `@property` + `@abstractmethod` (`Employee` above) | Same enforcement, but ALSO dictates the ACCESS style | You need every subclass to expose a specific VALUE via attribute syntax (`obj.x`), not a method call (`obj.x()`) |

> ❌ Misconception: `abc`'s only job is to stop you from creating the base class directly.
> ✅ Correct: that's a side effect. Its real job is guaranteeing every *concrete* subclass fully implements the contract too — as `Triangle` ([Part 1](15_Oops_part1.ipynb)) and `Intern` (above) both prove, an incomplete subclass is blocked just as hard as the abstract base class itself.

<a id="misconceptions"></a>
## ⚠ Common Misconceptions

❌ `__variable` in Python makes an attribute truly private, like Java's `private`.
✅ It only triggers **name mangling** (`_ClassName__variable`) — the data is still reachable if you know the mangled name. Python's model is "we're all consenting adults," relying on convention, not enforcement.

❌ `_variable` and `__variable` do the same thing.
✅ `_variable` is a pure convention (nothing happens). `__variable` actually gets renamed by the interpreter.

❌ `@property` getters/setters exist to make attributes "unreadable" from outside.
✅ Their real purpose is to run **validation or extra logic** on read/write while keeping the clean `obj.attr` syntax — not to block access.

❌ Passing an object into a function always protects the original from changes.
✅ Objects are passed by reference — mutating the object's attributes inside the function **does** affect the original. Only *reassigning* the local parameter name (or returning a new object without saving it back) leaves the original untouched.

❌ A missing `@staticmethod` decorator only matters stylistically.
✅ It can cause a real runtime crash when the method is called on an instance instead of the class, because Python then tries to auto-pass `self` as an extra argument.

❌ Python doesn't support multiple inheritance (only Java/C#-style single inheritance).
✅ Python fully supports multiple inheritance — `class Child(Father, Mother):` is valid and common. Java/C# are the ones that restrict classes to a single parent.

❌ When a subclass overrides a method, the parent's version is gone forever for that subclass.
✅ The parent's version still exists and can be called explicitly with `super().method()` — overriding just means the CHILD's version runs by default when you call `obj.method()`.

❌ If two parent classes define the same method, Python can't decide which one to use and either crashes or picks randomly.
✅ Python always resolves it deterministically through the **Method Resolution Order (MRO)** — a computable, inspectable order (`ClassName.mro()`) based on the order parents are listed.

❌ Defining `add(self, a, b)` and then `add(self, a, b, c)` in the same class overloads the method, like in Java.
✅ Python has no method overloading — the second definition completely replaces the first. Use default arguments (`c=0`) or `*args` to accept a variable number of parameters instead.

❌ Polymorphism in Python requires classes to share a common parent or interface.
✅ Duck typing means Python only checks that an object HAS the method being called, at the moment it's called — no shared ancestry required at all.

❌ `__str__` and `__repr__` are interchangeable — defining one covers both cases.
✅ `print()`/`str()` use `__str__`; containers (lists, dicts) and the console echo use `__repr__` for each item. Define both for complete coverage.

❌ Defining `__eq__` on a class has no side effects beyond changing how `==` works.
✅ It automatically sets `__hash__` to `None`, making the object unhashable (can't go in a `set` or be a `dict` key) unless you explicitly define `__hash__` too.

❌ An abstract base class (`ABC`) can only contain abstract methods — no real logic at all.
✅ It can freely mix concrete methods (like `log()` and `notify()` in `Notification`) alongside abstract ones — this is the Template Method pattern, and it's a completely normal, common design.

❌ `@abstractmethod` only prevents the BASE class from being instantiated.
✅ It also blocks any SUBCLASS that fails to implement every abstract member — `Triangle` (Part 1) and `Intern` (above) both prove a half-finished subclass is rejected just as hard as the abstract base itself.

<a id="interview-questions"></a>
## 🔍 Interview Questions

- Why doesn't Python have true private variables like Java or C++?
- What is name mangling, and what problem does it actually solve (hint: it's mostly about subclass name clashes, not security)?
- What's the difference between a traditional getter/setter method and a Python `@property`?
- If you pass an object into a function and mutate one of its attributes, does the caller see the change? What if you reassign the parameter to a brand-new object instead?
- What's the difference between a class variable and an instance variable? What happens if an `__init__` accidentally resets what should be a persistent counter?
- Why would forgetting `@staticmethod` cause a `TypeError` only when the method is called on an instance, but not when called on the class?
- What is the difference between Aggregation ("Has-A") and Inheritance ("Is-A")? Give an example of each.
- What are the four types of inheritance in Python? Give a one-line example of each.
- What is Method Resolution Order (MRO), and how does it solve the "diamond problem" in multiple inheritance?
- What's the difference between overriding a method that calls `super()` versus one that doesn't?
- What's the difference between `isinstance()` and `issubclass()`? Why is `issubclass(Employee, Manager)` `False` even though `issubclass(Manager, Employee)` is `True`?
- Why does Java disallow multiple inheritance for classes, but Python allows it? What mechanism does Python use to avoid the ambiguity that worried Java's designers?
- What is polymorphism, and how does Python achieve it differently than Java (hint: duck typing vs interfaces)?
- Why doesn't Python support method overloading the way Java does? How would you replicate similar behavior?
- What's the relationship between inheritance, method overriding, and polymorphism?
- What is a dunder method? Name at least four and what built-in syntax each one enables.
- What's the difference between `__str__` and `__repr__`? Which one does a list use when printing its items?
- Why does defining `__eq__` on a class make it unhashable unless you also define `__hash__`?
- What is the Template Method pattern, and how does mixing concrete and abstract methods in one `ABC` implement it?
- What does stacking `@property` under `@abstractmethod` actually enforce that a plain `@abstractmethod` doesn't?
- If a subclass implements 4 out of 5 abstract methods from its parent `ABC`, can you instantiate it? Why or why not?

<a id="key-takeaways"></a>
## 🎯 Key Takeaways

1. Python privacy is convention-based: `_x` means "please don't touch," `__x` triggers name mangling (`_ClassName__x`) — neither is a hard security boundary.
2. `@property` + `@x.setter` is the idiomatic way to add validation while keeping simple attribute-style syntax (`obj.x` / `obj.x = value`).
3. Objects (like lists, dicts, sets) are mutable and passed by reference — mutating an attribute inside a function is visible outside it, but reassigning the parameter or returning a new object is not.
4. Class variables are shared across all instances; resetting a value inside `__init__` (like `self.sno = 0`) instead of reading the class-level counter defeats the purpose of a persistent counter.
5. A missing `@staticmethod` can silently "work" when called via the class but crash with `TypeError` when called via an instance — always add it explicitly if `self` isn't needed.
6. Operator-overload methods (`__add__`, `__mul__`, etc.) should return a new instance of the *same class*, not a plain value like a string — otherwise chained operations break.
7. Aggregation ("Has-A": a class holds another as an attribute) and Inheritance ("Is-A": a subclass extends a parent) are the two fundamental ways classes relate to each other.
8. Python supports four shapes of inheritance — Single, Multilevel (a chain), Hierarchical (shared parent, multiple children), and Multiple (one child, several parents) — and unlike Java/C#, it allows true multiple inheritance for regular classes.
9. When multiple parents define the same method, Python resolves the conflict deterministically via **MRO** (`ClassName.mro()`), not randomly — the order parents are listed in the class definition decides the outcome.
10. `super()` lets an overriding method call the parent's original version instead of fully discarding it — the difference between *extending* a parent's behavior and *replacing* it entirely.
11. `isinstance(obj, Class)` checks an object against a class (including its ancestors); `issubclass(A, B)` checks the class relationship directly — and that relationship is one-directional (child → ancestor, never the reverse).
12. Polymorphism means the same method call behaves differently depending on the object — Python achieves this through **duck typing** (no shared ancestry required) as well as through **method overriding** (via inheritance).
13. Python has no method overloading like Java — redefining a method with different parameters just replaces the old version. Use default arguments or `*args` instead.
14. Dunder methods (`__init__`, `__str__`, `__add__`, `__eq__`, ...) are how a class plugs into Python's built-in syntax and functions — operator overloading is just one application of this general mechanism.
15. `__str__` powers `print()`/`str()`; `__repr__` powers containers and the console echo — define both, since only having `__str__` leaves lists of your objects looking unhelpful.
16. Defining `__eq__` without `__hash__` makes an object unhashable — always define both together, based on the same fields.
17. `ABC` + `@abstractmethod` can mix concrete and abstract methods in one base class (the Template Method pattern) — the parent defines the fixed algorithm and calls the not-yet-written piece from inside its own code.
18. Stacking `@property` under `@abstractmethod` forces every subclass to expose a VALUE via attribute syntax (`obj.x`), not just implement a method — the contract covers both existence AND access style.
19. `abc` enforcement isn't limited to blocking the base class — any subclass that implements only *some* of the required abstract methods is blocked from instantiation too, with the same `TypeError`.